In [1]:
import os
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# =========================================================================
# Configuration
# =========================================================================

PATH        = r'C:\Users\lidon\Desktop\2026spring\IDX_MLS_Analytics'
SOLD_CSV    = os.path.join(PATH, 'all_sold_cleaned.csv')
LISTING_CSV = os.path.join(PATH, 'all_listings_cleaned.csv')
DTYPE_SPEC  = {'PostalCode': str, 'ListingKey': str}

# =========================================================================
# Load cleaned datasets
# =========================================================================

print("Loading cleaned datasets...")
sold     = pd.read_csv(SOLD_CSV,    dtype=DTYPE_SPEC, low_memory=False)
listings = pd.read_csv(LISTING_CSV, dtype=DTYPE_SPEC, low_memory=False)

# Re-parse date columns (datetime is not preserved through CSV)
DATE_COLS = ['CloseDate', 'PurchaseContractDate',
             'ListingContractDate', 'ContractStatusChangeDate']
for df in [sold, listings]:
    for col in DATE_COLS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

print(f"  all_sold_cleaned     : {len(sold):,} rows  |  {sold.shape[1]} columns")
print(f"  all_listings_cleaned : {len(listings):,} rows  |  {listings.shape[1]} columns\n")

# =========================================================================
# STEP 1 — Validate source columns before engineering
# Check: missing values, zero/negative denominators, bad dtypes
# =========================================================================

print("=" * 65)
print("STEP 1 — PRE-ENGINEERING VALIDATION")
print("=" * 65)

RATIO_SOURCES = {
    'price_ratio':             ('ClosePrice', 'ListPrice'),
    'close_to_orig_list_ratio':('ClosePrice', 'OriginalListPrice'),
    'price_per_sqft':          ('ClosePrice', 'LivingArea'),
}
DATE_SOURCES = {
    'listing_to_contract_days': ('ListingContractDate', 'PurchaseContractDate'),
    'contract_to_close_days':   ('PurchaseContractDate', 'CloseDate'),
}

print("\n  Ratio source columns (sold):")
for metric, (num, den) in RATIO_SOURCES.items():
    num_null = sold[num].isna().sum() if num in sold.columns else 'N/A'
    den_null = sold[den].isna().sum() if den in sold.columns else 'N/A'
    den_zero = ((sold[den] <= 0).sum()
                if den in sold.columns else 'N/A')
    print(f"    {metric:<30} "
          f"num null={num_null:,}  "
          f"den null={den_null:,}  "
          f"den<=0={den_zero:,}")

print("\n  Date source columns (sold):")
for metric, (start, end) in DATE_SOURCES.items():
    start_null = sold[start].isna().sum() if start in sold.columns else 'N/A'
    end_null   = sold[end].isna().sum()   if end   in sold.columns else 'N/A'
    print(f"    {metric:<30} "
          f"start null={start_null:,}  "
          f"end null={end_null:,}")

# =========================================================================
# STEP 2 — Feature engineering
# All metrics computed on sold table only (ratios require ClosePrice)
# =========================================================================

print("\n" + "=" * 65)
print("STEP 2 — FEATURE ENGINEERING")
print("=" * 65)

df = sold.copy()

# ── 2a. Price Ratio: ClosePrice / ListPrice ───────────────────────────────
# Measures how close the final sale price was to the last listing price.
# > 1.0 means buyer paid above asking (overbid); < 1.0 means discount.
if 'ClosePrice' in df.columns and 'ListPrice' in df.columns:
    valid = df['ListPrice'].notna() & (df['ListPrice'] > 0)
    df['price_ratio'] = np.where(
        valid,
        df['ClosePrice'] / df['ListPrice'],
        np.nan
    )
    print(f"  price_ratio               "
          f"non-null: {df['price_ratio'].notna().sum():,}  "
          f"median: {df['price_ratio'].median():.4f}")

# ── 2b. Close-to-Original-List Ratio: ClosePrice / OriginalListPrice ──────
# Captures the full price reduction history from first listing to close.
if 'ClosePrice' in df.columns and 'OriginalListPrice' in df.columns:
    valid = df['OriginalListPrice'].notna() & (df['OriginalListPrice'] > 0)
    df['close_to_orig_list_ratio'] = np.where(
        valid,
        df['ClosePrice'] / df['OriginalListPrice'],
        np.nan
    )
    print(f"  close_to_orig_list_ratio  "
          f"non-null: {df['close_to_orig_list_ratio'].notna().sum():,}  "
          f"median: {df['close_to_orig_list_ratio'].median():.4f}")

# ── 2c. Price Per Square Foot: ClosePrice / LivingArea ────────────────────
# Normalizes price across different property sizes.
if 'ClosePrice' in df.columns and 'LivingArea' in df.columns:
    valid = df['LivingArea'].notna() & (df['LivingArea'] > 0)
    df['price_per_sqft'] = np.where(
        valid,
        df['ClosePrice'] / df['LivingArea'],
        np.nan
    )
    print(f"  price_per_sqft            "
          f"non-null: {df['price_per_sqft'].notna().sum():,}  "
          f"median: ${df['price_per_sqft'].median():.0f}")

# ── 2d. Days on Market — use existing field ────────────────────────────────
print(f"  DaysOnMarket (existing)   "
      f"non-null: {df['DaysOnMarket'].notna().sum():,}  "
      f"median: {df['DaysOnMarket'].median():.0f} days")

# ── 2e. Year / Month / YrMo from CloseDate ────────────────────────────────
if 'CloseDate' in df.columns:
    df['close_year']  = df['CloseDate'].dt.year
    df['close_month'] = df['CloseDate'].dt.month
    df['close_yrmo']  = df['CloseDate'].dt.to_period('M').astype(str)
    print(f"  close_year / close_month / close_yrmo  "
          f"non-null: {df['close_yrmo'].notna().sum():,}")

# ── 2f. Listing-to-Contract Days ──────────────────────────────────────────
# Time from listing to accepted offer (how fast the market moved).
if 'ListingContractDate' in df.columns and 'PurchaseContractDate' in df.columns:
    delta = (df['PurchaseContractDate'] - df['ListingContractDate']).dt.days
    df['listing_to_contract_days'] = np.where(delta >= 0, delta, np.nan)
    neg = (delta < 0).sum()
    print(f"  listing_to_contract_days  "
          f"non-null: {df['listing_to_contract_days'].notna().sum():,}  "
          f"median: {df['listing_to_contract_days'].median():.0f} days  "
          f"(negative → NaN: {neg:,})")

# ── 2g. Contract-to-Close Days ────────────────────────────────────────────
# Escrow and closing period duration.
if 'PurchaseContractDate' in df.columns and 'CloseDate' in df.columns:
    delta = (df['CloseDate'] - df['PurchaseContractDate']).dt.days
    df['contract_to_close_days'] = np.where(delta >= 0, delta, np.nan)
    neg = (delta < 0).sum()
    print(f"  contract_to_close_days    "
          f"non-null: {df['contract_to_close_days'].notna().sum():,}  "
          f"median: {df['contract_to_close_days'].median():.0f} days  "
          f"(negative → NaN: {neg:,})")

sold = df

# =========================================================================
# STEP 3 — Sample output table
# Show first 10 rows of all newly created metric columns
# =========================================================================

print("\n" + "=" * 65)
print("STEP 3 — SAMPLE OUTPUT TABLE (first 10 rows)")
print("=" * 65)

NEW_COLS = [
    'ListingKey', 'CountyOrParish', 'CloseDate',
    'ClosePrice', 'ListPrice', 'OriginalListPrice', 'LivingArea',
    'price_ratio', 'close_to_orig_list_ratio', 'price_per_sqft',
    'DaysOnMarket', 'close_year', 'close_month', 'close_yrmo',
    'listing_to_contract_days', 'contract_to_close_days',
]
show_cols = [c for c in NEW_COLS if c in sold.columns]
print(sold[show_cols].head(10).to_string(index=False))

# =========================================================================
# STEP 4 — School district spatial join
# Requires: pip install geopandas
# GeoJSON from: https://data.ca.gov/dataset/california-school-district-areas-2025-26
# =========================================================================

print("\n" + "=" * 65)
print("STEP 4 — SCHOOL DISTRICT SPATIAL JOIN")
print("=" * 65)

GEOJSON_PATH = os.path.join(PATH, 'california_school_districts_2025.geojson')

if not os.path.exists(GEOJSON_PATH):
    print(f"  GeoJSON not found at: {GEOJSON_PATH}")
    print("  Download from: https://data.ca.gov/dataset/california-school-district-areas-2025-26")
    print("  Save as 'california_school_districts_2025.geojson' in your project folder.")
    print("  Then re-run this script.")
else:
    try:
        import geopandas as gpd
        from shapely.geometry import Point

        print("  Loading school district boundaries...")
        gdf_districts = gpd.read_file(GEOJSON_PATH)

        # Filter to Unified school districts only
        gdf_districts = gdf_districts[
            gdf_districts['DistrictType'] == 'Unified'
        ].copy()
        print(f"  Unified districts: {len(gdf_districts)}")

        # Only join rows with valid, in-CA coordinates
        valid_geo = (
            sold['Latitude'].notna()  & sold['Longitude'].notna() &
            ~sold['geo_missing_flag'] & ~sold['lat_0_flag'] &
            ~sold['out_of_state_flag']
        ) if 'geo_missing_flag' in sold.columns else (
            sold['Latitude'].notna() & sold['Longitude'].notna()
        )

        sold_geo = sold[valid_geo].copy()
        sold_geo['geometry'] = sold_geo.apply(
            lambda row: Point(row['Longitude'], row['Latitude']), axis=1
        )
        sold_gdf = gpd.GeoDataFrame(sold_geo, geometry='geometry',
                                     crs='EPSG:4326')

        # Ensure both GeoDataFrames use the same CRS
        gdf_districts = gdf_districts.to_crs('EPSG:4326')

        print(f"  Running spatial join on {len(sold_gdf):,} valid-coordinate rows...")
        joined = gpd.sjoin(
            sold_gdf[['ListingKey', 'geometry']],
            gdf_districts[['DistrictName', 'geometry']],
            how='left',
            predicate='within'
        )

        # Merge DistrictName back to the full sold table
        sold = sold.merge(
            joined[['ListingKey', 'DistrictName']].drop_duplicates('ListingKey'),
            on='ListingKey', how='left'
        )

        matched = sold['DistrictName'].notna().sum()
        print(f"  Matched to a district : {matched:,} / {len(sold):,} "
              f"({matched/len(sold)*100:.1f}%)")
        print(f"  Sample district names : "
              f"{sold['DistrictName'].dropna().unique()[:5].tolist()}")

    except ImportError:
        print("  geopandas not installed. Run: pip install geopandas")
    except Exception as e:
        print(f"  Spatial join failed: {e}")

# =========================================================================
# STEP 5 — Segment analysis
# =========================================================================

print("\n" + "=" * 65)
print("STEP 5 — SEGMENT ANALYSIS")
print("=" * 65)

METRIC_COLS = [
    'ClosePrice', 'price_per_sqft', 'DaysOnMarket',
    'price_ratio', 'close_to_orig_list_ratio',
    'listing_to_contract_days', 'contract_to_close_days',
]

def segment_summary(df, group_col, label):
    if group_col not in df.columns:
        print(f"\n  [{label}] '{group_col}' not found — skipped")
        return
    available = [c for c in METRIC_COLS if c in df.columns]
    agg = {c: ['count', 'median', 'mean'] for c in available}
    # Use count on ClosePrice to get number of sales
    summary = (
        df.groupby(group_col)[available]
        .agg(['count', 'median', 'mean'])
    )
    # Flatten multi-level columns
    summary.columns = ['_'.join(c) for c in summary.columns]
    summary = summary.sort_values('ClosePrice_count', ascending=False)
    print(f"\n  [{label}] grouped by {group_col} (top 15):")
    print(summary.head(15).to_string(float_format=lambda x: f'{x:,.1f}'))

# 5a. By PropertyType and PropertySubType
segment_summary(sold, 'PropertyType',    'PropertyType')
segment_summary(sold, 'PropertySubType', 'PropertySubType')

# 5b. By CountyOrParish and MLSAreaMajor
segment_summary(sold, 'CountyOrParish', 'CountyOrParish')
segment_summary(sold, 'MLSAreaMajor',   'MLSAreaMajor')

# 5c. By ListOfficeName and BuyerOfficeName (competitive intelligence)
segment_summary(sold, 'ListOfficeName',  'ListOfficeName (top agents by volume)')
segment_summary(sold, 'BuyerOfficeName', 'BuyerOfficeName (top buyer offices)')

# =========================================================================
# Save enriched sold dataset
# =========================================================================

print("\n" + "=" * 65)
print("SAVING ENRICHED DATASET")
print("=" * 65)

out_path = os.path.join(PATH, 'all_sold_features.csv')
sold.to_csv(out_path, index=False, encoding='utf-8')
print(f"  Saved: all_sold_features.csv — "
      f"{len(sold):,} rows  |  {sold.shape[1]} columns")

new_metric_cols = [
    'price_ratio', 'close_to_orig_list_ratio', 'price_per_sqft',
    'close_year', 'close_month', 'close_yrmo',
    'listing_to_contract_days', 'contract_to_close_days',
]
if 'DistrictName' in sold.columns:
    new_metric_cols.append('DistrictName')
print(f"\n  New columns added: {new_metric_cols}")

print("\n" + "=" * 65)
print("WEEK 6 FEATURE ENGINEERING COMPLETE")
print("=" * 65)

Loading cleaned datasets...
  all_sold_cleaned     : 447,769 rows  |  76 columns
  all_listings_cleaned : 615,316 rows  |  67 columns

STEP 1 — PRE-ENGINEERING VALIDATION

  Ratio source columns (sold):
    price_ratio                    num null=2  den null=0  den<=0=0
    close_to_orig_list_ratio       num null=2  den null=822  den<=0=2
    price_per_sqft                 num null=2  den null=253  den<=0=0

  Date source columns (sold):
    listing_to_contract_days       start null=1  end null=198
    contract_to_close_days         start null=198  end null=0

STEP 2 — FEATURE ENGINEERING
  price_ratio               non-null: 447,767  median: 1.0000
  close_to_orig_list_ratio  non-null: 446,943  median: 0.9957
  price_per_sqft            non-null: 447,514  median: $537
  DaysOnMarket (existing)   non-null: 447,769  median: 18 days
  close_year / close_month / close_yrmo  non-null: 447,769
  listing_to_contract_days  non-null: 447,281  median: 25 days  (negative → NaN: 289)
  contract_t

In [2]:
import os
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# =========================================================================
# Configuration
# =========================================================================

PATH        = r'C:\Users\lidon\Desktop\2026spring\IDX_MLS_Analytics'
SOLD_CSV    = os.path.join(PATH, 'all_sold_cleaned.csv')
LISTING_CSV = os.path.join(PATH, 'all_listings_cleaned.csv')
DTYPE_SPEC  = {'PostalCode': str, 'ListingKey': str}

# =========================================================================
# Load cleaned datasets
# =========================================================================

print("Loading cleaned datasets...")
sold     = pd.read_csv(SOLD_CSV,    dtype=DTYPE_SPEC, low_memory=False)
listings = pd.read_csv(LISTING_CSV, dtype=DTYPE_SPEC, low_memory=False)

# Re-parse date columns (datetime is not preserved through CSV)
DATE_COLS = ['CloseDate', 'PurchaseContractDate',
             'ListingContractDate', 'ContractStatusChangeDate']
for df in [sold, listings]:
    for col in DATE_COLS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

print(f"  all_sold_cleaned     : {len(sold):,} rows  |  {sold.shape[1]} columns")
print(f"  all_listings_cleaned : {len(listings):,} rows  |  {listings.shape[1]} columns\n")

# =========================================================================
# STEP 1 — Validate source columns before engineering
# Check: missing values, zero/negative denominators, bad dtypes
# =========================================================================

print("=" * 65)
print("STEP 1 — PRE-ENGINEERING VALIDATION")
print("=" * 65)

RATIO_SOURCES = {
    'price_ratio':             ('ClosePrice', 'ListPrice'),
    'close_to_orig_list_ratio':('ClosePrice', 'OriginalListPrice'),
    'price_per_sqft':          ('ClosePrice', 'LivingArea'),
}
DATE_SOURCES = {
    'listing_to_contract_days': ('ListingContractDate', 'PurchaseContractDate'),
    'contract_to_close_days':   ('PurchaseContractDate', 'CloseDate'),
}

print("\n  Ratio source columns (sold):")
for metric, (num, den) in RATIO_SOURCES.items():
    num_null = sold[num].isna().sum() if num in sold.columns else 'N/A'
    den_null = sold[den].isna().sum() if den in sold.columns else 'N/A'
    den_zero = ((sold[den] <= 0).sum()
                if den in sold.columns else 'N/A')
    print(f"    {metric:<30} "
          f"num null={num_null:,}  "
          f"den null={den_null:,}  "
          f"den<=0={den_zero:,}")

print("\n  Date source columns (sold):")
for metric, (start, end) in DATE_SOURCES.items():
    start_null = sold[start].isna().sum() if start in sold.columns else 'N/A'
    end_null   = sold[end].isna().sum()   if end   in sold.columns else 'N/A'
    print(f"    {metric:<30} "
          f"start null={start_null:,}  "
          f"end null={end_null:,}")

# =========================================================================
# STEP 2 — Feature engineering
# All metrics computed on sold table only (ratios require ClosePrice)
# =========================================================================

print("\n" + "=" * 65)
print("STEP 2 — FEATURE ENGINEERING")
print("=" * 65)

df = sold.copy()

# ── 2a. Price Ratio: ClosePrice / ListPrice ───────────────────────────────
# Measures how close the final sale price was to the last listing price.
# > 1.0 means buyer paid above asking (overbid); < 1.0 means discount.
if 'ClosePrice' in df.columns and 'ListPrice' in df.columns:
    valid = df['ListPrice'].notna() & (df['ListPrice'] > 0)
    df['price_ratio'] = np.where(
        valid,
        df['ClosePrice'] / df['ListPrice'],
        np.nan
    )
    print(f"  price_ratio               "
          f"non-null: {df['price_ratio'].notna().sum():,}  "
          f"median: {df['price_ratio'].median():.4f}")

# ── 2b. Close-to-Original-List Ratio: ClosePrice / OriginalListPrice ──────
# Captures the full price reduction history from first listing to close.
if 'ClosePrice' in df.columns and 'OriginalListPrice' in df.columns:
    valid = df['OriginalListPrice'].notna() & (df['OriginalListPrice'] > 0)
    df['close_to_orig_list_ratio'] = np.where(
        valid,
        df['ClosePrice'] / df['OriginalListPrice'],
        np.nan
    )
    print(f"  close_to_orig_list_ratio  "
          f"non-null: {df['close_to_orig_list_ratio'].notna().sum():,}  "
          f"median: {df['close_to_orig_list_ratio'].median():.4f}")

# ── 2c. Price Per Square Foot: ClosePrice / LivingArea ────────────────────
# Normalizes price across different property sizes.
if 'ClosePrice' in df.columns and 'LivingArea' in df.columns:
    valid = df['LivingArea'].notna() & (df['LivingArea'] > 0)
    df['price_per_sqft'] = np.where(
        valid,
        df['ClosePrice'] / df['LivingArea'],
        np.nan
    )
    print(f"  price_per_sqft            "
          f"non-null: {df['price_per_sqft'].notna().sum():,}  "
          f"median: ${df['price_per_sqft'].median():.0f}")

# ── 2d. Days on Market — use existing field ────────────────────────────────
print(f"  DaysOnMarket (existing)   "
      f"non-null: {df['DaysOnMarket'].notna().sum():,}  "
      f"median: {df['DaysOnMarket'].median():.0f} days")

# ── 2e. Year / Month / YrMo from CloseDate ────────────────────────────────
if 'CloseDate' in df.columns:
    df['close_year']  = df['CloseDate'].dt.year
    df['close_month'] = df['CloseDate'].dt.month
    df['close_yrmo']  = df['CloseDate'].dt.to_period('M').astype(str)
    print(f"  close_year / close_month / close_yrmo  "
          f"non-null: {df['close_yrmo'].notna().sum():,}")

# ── 2f. Listing-to-Contract Days ──────────────────────────────────────────
# Time from listing to accepted offer (how fast the market moved).
if 'ListingContractDate' in df.columns and 'PurchaseContractDate' in df.columns:
    delta = (df['PurchaseContractDate'] - df['ListingContractDate']).dt.days
    df['listing_to_contract_days'] = np.where(delta >= 0, delta, np.nan)
    neg = (delta < 0).sum()
    print(f"  listing_to_contract_days  "
          f"non-null: {df['listing_to_contract_days'].notna().sum():,}  "
          f"median: {df['listing_to_contract_days'].median():.0f} days  "
          f"(negative → NaN: {neg:,})")

# ── 2g. Contract-to-Close Days ────────────────────────────────────────────
# Escrow and closing period duration.
if 'PurchaseContractDate' in df.columns and 'CloseDate' in df.columns:
    delta = (df['CloseDate'] - df['PurchaseContractDate']).dt.days
    df['contract_to_close_days'] = np.where(delta >= 0, delta, np.nan)
    neg = (delta < 0).sum()
    print(f"  contract_to_close_days    "
          f"non-null: {df['contract_to_close_days'].notna().sum():,}  "
          f"median: {df['contract_to_close_days'].median():.0f} days  "
          f"(negative → NaN: {neg:,})")

sold = df

# =========================================================================
# STEP 3 — Sample output table
# Show first 10 rows of all newly created metric columns
# =========================================================================

print("\n" + "=" * 65)
print("STEP 3 — SAMPLE OUTPUT TABLE (first 10 rows)")
print("=" * 65)

NEW_COLS = [
    'ListingKey', 'CountyOrParish', 'CloseDate',
    'ClosePrice', 'ListPrice', 'OriginalListPrice', 'LivingArea',
    'price_ratio', 'close_to_orig_list_ratio', 'price_per_sqft',
    'DaysOnMarket', 'close_year', 'close_month', 'close_yrmo',
    'listing_to_contract_days', 'contract_to_close_days',
]
show_cols = [c for c in NEW_COLS if c in sold.columns]
print(sold[show_cols].head(10).to_string(index=False))

# =========================================================================
# STEP 4 — School district spatial join
# Requires: pip install geopandas
# GeoJSON from: https://data.ca.gov/dataset/california-school-district-areas-2025-26
# =========================================================================

print("\n" + "=" * 65)
print("STEP 4 — SCHOOL DISTRICT SPATIAL JOIN")
print("=" * 65)

GEOJSON_PATH = os.path.join(PATH, 'california_school_districts_2025.geojson')

if not os.path.exists(GEOJSON_PATH):
    print(f"  GeoJSON not found at: {GEOJSON_PATH}")
    print("  Download from: https://data.ca.gov/dataset/california-school-district-areas-2025-26")
    print("  Save as 'california_school_districts_2025.geojson' in your project folder.")
    print("  Then re-run this script.")
else:
    try:
        import geopandas as gpd
        from shapely.geometry import Point

        print("  Loading school district boundaries...")
        gdf_districts = gpd.read_file(GEOJSON_PATH)

        # Filter to Unified school districts only
        gdf_districts = gdf_districts[
            gdf_districts['DistrictType'] == 'Unified'
        ].copy()
        print(f"  Unified districts: {len(gdf_districts)}")

        # Only join rows with valid, in-CA coordinates
        valid_geo = (
            sold['Latitude'].notna()  & sold['Longitude'].notna() &
            ~sold['geo_missing_flag'] & ~sold['lat_0_flag'] &
            ~sold['out_of_state_flag']
        ) if 'geo_missing_flag' in sold.columns else (
            sold['Latitude'].notna() & sold['Longitude'].notna()
        )

        sold_geo = sold[valid_geo].copy()
        sold_geo['geometry'] = sold_geo.apply(
            lambda row: Point(row['Longitude'], row['Latitude']), axis=1
        )
        sold_gdf = gpd.GeoDataFrame(sold_geo, geometry='geometry',
                                     crs='EPSG:4326')

        # Ensure both GeoDataFrames use the same CRS
        gdf_districts = gdf_districts.to_crs('EPSG:4326')

        print(f"  Running spatial join on {len(sold_gdf):,} valid-coordinate rows...")
        joined = gpd.sjoin(
            sold_gdf[['ListingKey', 'geometry']],
            gdf_districts[['DistrictName', 'geometry']],
            how='left',
            predicate='within'
        )

        # Merge DistrictName back to the full sold table
        sold = sold.merge(
            joined[['ListingKey', 'DistrictName']].drop_duplicates('ListingKey'),
            on='ListingKey', how='left'
        )

        matched = sold['DistrictName'].notna().sum()
        print(f"  Matched to a district : {matched:,} / {len(sold):,} "
              f"({matched/len(sold)*100:.1f}%)")
        print(f"  Sample district names : "
              f"{sold['DistrictName'].dropna().unique()[:5].tolist()}")

    except ImportError:
        print("  geopandas not installed. Run: pip install geopandas")
    except Exception as e:
        print(f"  Spatial join failed: {e}")

# =========================================================================
# STEP 5 — Segment analysis
# =========================================================================

print("\n" + "=" * 65)
print("STEP 5 — SEGMENT ANALYSIS")
print("=" * 65)

def segment_summary(df, group_col, label, top_n=15):
    if group_col not in df.columns:
        print(f"\n  [{label}] '{group_col}' not found — skipped")
        return

    # Define aggregations with short readable names
    agg_map = {}
    col_rename = {}

    pairs = [
        ('ClosePrice',               'sales',      'median_price',   'avg_price'),
        ('price_per_sqft',           'n_ppsf',     'median_ppsf',    'avg_ppsf'),
        ('DaysOnMarket',             'n_dom',      'median_dom',     'avg_dom'),
        ('price_ratio',              'n_pr',       'median_pr',      'avg_pr'),
        ('close_to_orig_list_ratio', 'n_colr',     'median_colr',    'avg_colr'),
        ('listing_to_contract_days', 'n_l2c',      'median_l2c',     'avg_l2c'),
        ('contract_to_close_days',   'n_c2cl',     'median_c2cl',    'avg_c2cl'),
    ]

    for col, cnt_name, med_name, avg_name in pairs:
        if col not in df.columns:
            continue
        agg_map[col] = ['count', 'median', 'mean']
        col_rename[(col, 'count')]  = cnt_name
        col_rename[(col, 'median')] = med_name
        col_rename[(col, 'mean')]   = avg_name

    if not agg_map:
        return

    summary = df.groupby(group_col).agg(agg_map)
    summary.columns = [col_rename.get(c, '_'.join(c))
                       for c in summary.columns]
    summary = summary.sort_values('sales', ascending=False)

    # Keep only the most meaningful columns for display
    display_cols = [c for c in [
        'sales', 'median_price', 'avg_price',
        'median_ppsf', 'median_dom',
        'median_pr', 'median_colr',
        'median_l2c', 'median_c2cl',
    ] if c in summary.columns]

    top = summary[display_cols].head(top_n)

    # Format numbers for readability
    fmt = {}
    for c in top.columns:
        if 'price' in c or 'ppsf' in c:
            fmt[c] = lambda x: f'${x:,.0f}' if pd.notna(x) else '-'
        elif c == 'sales':
            fmt[c] = lambda x: f'{x:,.0f}'
        else:
            fmt[c] = lambda x: f'{x:.2f}' if pd.notna(x) else '-'

    # Print as aligned table
    print(f"\n  ── {label} (top {top_n} by sales) ──")
    header = f"  {'':45}" + "  ".join(f"{c:>13}" for c in display_cols)
    print(header)
    print("  " + "-" * (45 + 15 * len(display_cols)))
    for idx, row in top.iterrows():
        name = str(idx)[:44].ljust(45)
        vals = "  ".join(
            f"{'$'+f'{row[c]:,.0f}':>13}"
            if ('price' in c or 'ppsf' in c) and pd.notna(row[c])
            else f"{row[c]:>13,.0f}"
            if c == 'sales' and pd.notna(row[c])
            else f"{row[c]:>13.2f}"
            if pd.notna(row[c])
            else f"{'—':>13}"
            for c in display_cols
        )
        print(f"  {name}{vals}")

# 5a. PropertyType and PropertySubType
segment_summary(sold, 'PropertyType',    'By PropertyType')
segment_summary(sold, 'PropertySubType', 'By PropertySubType')

# 5b. CountyOrParish and MLSAreaMajor
segment_summary(sold, 'CountyOrParish', 'By County')
segment_summary(sold, 'MLSAreaMajor',   'By MLSAreaMajor')

# 5c. Competitive intelligence
segment_summary(sold, 'ListOfficeName',  'By Listing Office (competitive)')
segment_summary(sold, 'BuyerOfficeName', 'By Buyer Office (competitive)')

# =========================================================================
# Save enriched sold dataset
# =========================================================================

print("\n" + "=" * 65)
print("SAVING ENRICHED DATASET")
print("=" * 65)

out_path = os.path.join(PATH, 'all_sold_features.csv')
sold.to_csv(out_path, index=False, encoding='utf-8')
print(f"  Saved: all_sold_features.csv — "
      f"{len(sold):,} rows  |  {sold.shape[1]} columns")

new_metric_cols = [
    'price_ratio', 'close_to_orig_list_ratio', 'price_per_sqft',
    'close_year', 'close_month', 'close_yrmo',
    'listing_to_contract_days', 'contract_to_close_days',
]
if 'DistrictName' in sold.columns:
    new_metric_cols.append('DistrictName')
print(f"\n  New columns added: {new_metric_cols}")

print("\n" + "=" * 65)
print("WEEK 6 FEATURE ENGINEERING COMPLETE")
print("=" * 65)

Loading cleaned datasets...
  all_sold_cleaned     : 447,769 rows  |  76 columns
  all_listings_cleaned : 615,316 rows  |  67 columns

STEP 1 — PRE-ENGINEERING VALIDATION

  Ratio source columns (sold):
    price_ratio                    num null=2  den null=0  den<=0=0
    close_to_orig_list_ratio       num null=2  den null=822  den<=0=2
    price_per_sqft                 num null=2  den null=253  den<=0=0

  Date source columns (sold):
    listing_to_contract_days       start null=1  end null=198
    contract_to_close_days         start null=198  end null=0

STEP 2 — FEATURE ENGINEERING
  price_ratio               non-null: 447,767  median: 1.0000
  close_to_orig_list_ratio  non-null: 446,943  median: 0.9957
  price_per_sqft            non-null: 447,514  median: $537
  DaysOnMarket (existing)   non-null: 447,769  median: 18 days
  close_year / close_month / close_yrmo  non-null: 447,769
  listing_to_contract_days  non-null: 447,281  median: 25 days  (negative → NaN: 289)
  contract_t